# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print("Dataset Name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Identifier:", dataset.metadata.identifier)
print("License:", dataset.metadata.license)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's print out the record sets and their fields. All entities will be referenced by their `@id`.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets.keys())
for rs_id in record_sets:
    recset = dataset.record_sets[rs_id]
    print(f"RecordSet @id: {rs_id}, name: {recset.name if hasattr(recset, 'name') else 'N/A'}")
    print("  Fields:")
    for f in recset.fields:
        print(f"    Field @id: {f['@id']} | name: {f['name']}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here we demonstrate extracting all records for each record set, referencing entities by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for RecordSet {rs_id}:")
    print(df.columns.tolist())
    print(df.head(), "\n")

# For demonstration, use the first record set and its fields for further steps
first_rs_id = record_sets[0]
df = dataframes[first_rs_id]
print(f"Using RecordSet {first_rs_id} for EDA.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
All fields are referenced by their `@id`. Let's select a numeric field for demonstration based on the record set and its fields.

In [ ]:
# Choose a numeric field id; e.g., GAD-7 score field
# Let's look for a column with 'gad_7_score' or similar. If not, pick any numeric field.
numeric_fields = [col for col in df.columns if 'gad_7' in col.lower() or 'phq_9' in col.lower() or df[col].dtype in ['int64', 'float64']]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback: first column

print(f"Using numeric field @id: {numeric_field_id}")

# Filter records with numeric field > threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic field; e.g., 'gender' or similar
group_fields = [col for col in df.columns if 'gender' in col.lower()]
if group_fields:
    group_field_id = group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No demographic grouping field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
We'll plot the distribution of the selected numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped by a demographic field:
if group_fields:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, provides structured records with demographic and mental health indicator fields.
- We reviewed available record sets and fields referenced by their `@id`.
- A numeric field (e.g., GAD-7) was filtered and normalized, and demographic grouping analyzed.
- Visualizations reveal distributions and potential differences across demographic groups.
- This Croissant-based approach facilitates transparent, reproducible exploration using `mlcroissant`.